# Lab 3: N-gram language models

Guidelines: https://github.com/MohammedMosuily/NLP_Lab3

Run cells from top to bottom. Dataset paths are relative to this folder.

## Load and clean tweets
Remove hashtags, RT markers, URLs, mentions, and emojis. Each tweet is a separate sentence.
The CSV contains some invalid UTF-8 bytes; replace those characters while reading. Keep words from all languages in the corpus.

In [1]:
from pathlib import Path
import re
import emoji
import kagglehub
import pandas as pd
from nltk.lm import MLE
from nltk.lm.preprocessing import padded_everygram_pipeline, pad_both_ends
from nltk.util import bigrams
from sklearn.model_selection import train_test_split

path = Path(kagglehub.dataset_download("adizafar/large-random-tweets-from-pakistan"))
print([p.name for p in path.rglob("*.csv")])

['Random Tweets from Pakistan- Cleaned- Anonymous.csv']


In [2]:
df = pd.read_csv(path / "Random Tweets from Pakistan- Cleaned- Anonymous.csv", encoding_errors="replace", usecols=["full_text"])
print(df.columns.tolist())

['full_text']


In [3]:
def clean_tweet(text):
    text = re.sub(r"https?://\S+|www\.\S+", "", text)
    text = re.sub(r"[#@]\w+|\bRT\b", "", text)
    text = emoji.replace_emoji(text, replace="")
    return re.findall(r"[^\W\d_]+", text.lower())

tweets = [clean_tweet(text) for text in df["full_text"].dropna().astype(str)]
tweets = [tokens for tokens in tweets if tokens]
train_tweets, test_tweets = train_test_split(tweets, test_size=0.2, random_state=42)
train, vocab = padded_everygram_pipeline(2, train_tweets)
model = MLE(2)
model.fit(train, vocab)
print("Training tweets:", len(train_tweets))
print("Testing tweets:", len(test_tweets))

Training tweets: 143650
Testing tweets: 35913


## Evaluate and generate
An unsmoothed MLE model assigns zero probability to unseen bigrams, so held-out perplexity may be infinite. This is an expected limitation. A single word has no bigram context, so the requested perplexity of `pakistan` is calculated as a unigram, equal to `1 / P(pakistan)`.

In [4]:
test_bigrams = [pair for tweet in test_tweets for pair in bigrams(pad_both_ends(tweet, n=2))]
print("Test perplexity:", model.perplexity(test_bigrams))
print("P(is | pakistan):", model.score("is", ["pakistan"]))
print("Perplexity of pakistan:", model.perplexity([("pakistan",)]))
generated = model.generate(20, text_seed=["pakistan"], random_seed=42)
print("Generated tweet:", " ".join(["pakistan"] + generated).split("</s>")[0].strip())

Test perplexity: inf
P(is | pakistan): 0.05025704662293919
Perplexity of pakistan: 242.34320156000715


Generated tweet: pakistan now
